# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maheen-armghan/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [13]:
import os
if not os.path.exists("flyrank-internship"):
    !git clone https://github.com/maheen-armghan/flyrank-internship.git
os.chdir("flyrank-internship")
print("Now in:", os.getcwd())

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 343, done.
remote: Counting objects: 100% (343/343), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 343 (delta 188), reused 293 (delta 156), pack-reused 0 (from 0)
Receiving objects: 100% (343/343), 1.93 MiB | 2.69 MiB/s, done.
Resolving deltas: 100% (188/188), done.
Now in: /content/flyrank-internship/flyrank-internship/flyrank-internship


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates(subset="content_id")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("Rows:", len(df))

Rows: 30000


**Signal 1 — staleness vs decline: FALSE.** Decline rate does not increase cleanly with staleness (51.1% → 58.9% → 61.1% → 46.7% → 60.0% across buckets) — no monotonic trend, and the largest buckets (0-30 days, 90-180 days) both sit in the low-60s regardless of freshness. This matches Week 1's finding that median staleness was identical for declining vs. non-declining pages. FlyRank's staleness-based refresh flag isn't well-supported by this data alone.

**Signal 2 — CTR vs position tier: CONFIRMED.** Mean CTR drops cleanly and monotonically from 1.484 (top_3) to 0.150 (deep) — roughly a 10x difference across tiers. This strongly validates the assumption behind FlyRank's CTR-fix logic: comparing a page's CTR against its own tier's norm is meaningful, since tiers behave very differently from each other.

**Rule reasoning:** Since staleness is a weak/false signal on its own, my baseline rule will NOT weight it. Since position-adjusted CTR is confirmed strong, and demand (impressions) plus decline status are the clearest predictive signals from Week 1, my rule combines decline status with demand volume rather than staleness.

In [15]:
# Signal 1: staleness vs decline
bucket1 = df.groupby(pd.cut(df["days_since_last_update"], bins=[0,30,90,180,365,10000]))["is_declining_label"].agg(["mean","count"])
print(bucket1)
print("\nVerdict: FALSE — staleness bucket means look flat/inconsistent, matching Week 1's identical-median finding.")

                            mean  count
days_since_last_update                 
(0, 30]                 0.511377  20480
(30, 90]                0.588571    175
(90, 180]               0.611057   9171
(180, 365]              0.467456    169
(365, 10000]            0.600000      5

Verdict: FALSE — staleness bucket means look flat/inconsistent, matching Week 1's identical-median finding.


/tmp/ipykernel_921/3153594881.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket1 = df.groupby(pd.cut(df["days_since_last_update"], bins=[0,30,90,180,365,10000]))["is_declining_label"].agg(["mean","count"])


In [16]:
# Signal 2: CTR vs position tier
bucket2 = df.groupby("position_tier")["ctr"].agg(["mean","count"]).sort_index()
print(bucket2)
print("\nVerdict: CONFIRMED if CTR clearly drops as position tier worsens (check your real output).")

                   mean  count
position_tier                 
deep           0.150212   1319
page_1         0.652467  11814
page_3_5       0.222484   7242
striking       0.323239   7304
top_3          1.483611   2321

Verdict: CONFIRMED if CTR clearly drops as position tier worsens (check your real output).


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Rule:** `baseline_score = impressions_90d` when the page is both declining and has real demand (`impressions_90d >= 100`), else 0.
**Reason code:** `declining_with_demand`
**Action:** `review_for_refresh`

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
os.makedirs("work/outputs", exist_ok=True)

df["baseline_score"] = ((df["is_declining_label"] == 1) & (df["impressions_90d"] >= 100)).astype(int) * df["impressions_90d"]
df["reason_code"] = "declining_with_demand"
df["action"] = "review_for_refresh"

queue = df.sort_values("baseline_score", ascending=False)
queue[["content_id","baseline_score","reason_code","action"]].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Saved", len(queue), "rows to work/outputs/baseline_action_score.csv")
queue[["content_id","baseline_score","reason_code","action"]].head(10)

Saved 30000 rows to work/outputs/baseline_action_score.csv


,content_id,baseline_score,reason_code,action
6653,content_5fe46e04994d,517715,declining_with_demand,review_for_refresh
26844,content_8c19996aa890,509252,declining_with_demand,review_for_refresh
21819,content_4c36c775b818,463103,declining_with_demand,review_for_refresh
29879,content_1a9e894be2e2,416180,declining_with_demand,review_for_refresh
13537,content_2c2606c5d176,347399,declining_with_demand,review_for_refresh
26531,content_cb112fce36be,309910,declining_with_demand,review_for_refresh
21565,content_9532f197bbc8,309192,declining_with_demand,review_for_refresh
27478,content_008fb02c46cb,236803,declining_with_demand,review_for_refresh
23767,content_813e88069237,233561,declining_with_demand,review_for_refresh
26304,content_ff94c9b6b411,228566,declining_with_demand,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(20)[["content_id","impressions_90d","avg_position","ctr","trend_direction","baseline_score"]]
top10

,content_id,impressions_90d,avg_position,ctr,trend_direction,baseline_score
6653,content_5fe46e04994d,517715,4.2,0.14,down,517715
26844,content_8c19996aa890,509252,2.5,0.15,down,509252
21819,content_4c36c775b818,463103,2.3,0.41,down,463103
29879,content_1a9e894be2e2,416180,4.0,0.23,down,416180
13537,content_2c2606c5d176,347399,4.2,0.53,down,347399
26531,content_cb112fce36be,309910,5.6,0.16,down,309910
21565,content_9532f197bbc8,309192,2.0,0.87,down,309192
27478,content_008fb02c46cb,236803,4.4,0.26,down,236803
23767,content_813e88069237,233561,26.2,0.06,down,233561
26304,content_ff94c9b6b411,228566,27.4,0.04,down,228566


1. content_5fe46e04994d — 517,715 impressions, position 4.2, CTR 0.14, declining. Flagged for the highest demand in the dataset combined with confirmed decline. Would be wrong if this drop is seasonal or a temporary SERP change rather than a lasting decline.

2. content_8c19996aa890 — 509,252 impressions, position 2.5, CTR 0.15, declining. Strong position (top_3 tier) with unusually low CTR for that tier (expected ~1.48 per Signal 2) — likely a title/snippet issue, not a ranking problem. Would be wrong if this is a temporary SERP feature (e.g. featured snippet by a competitor) suppressing clicks rather than a real content issue.

3. content_4c36c775b818 — 463,103 impressions, position 2.3, CTR 0.41, declining. Also top-tier position, closer-to-expected CTR. Would be wrong if the decline is due to a sibling page cannibalizing traffic rather than genuine loss.

4. content_1a9e894be2e2 — 416,180 impressions, position 4.0, CTR 0.23, declining. High demand, page_1 tier position. Would be wrong if position is naturally volatile and this is normal day-to-day fluctuation, not a trend.

5. content_2c2606c5d176 — 347,399 impressions, position 4.2, CTR 0.53, declining. CTR is reasonably strong here, so the decline is likely driven by demand/position shifts rather than a click-through problem. Would be wrong if search_volume for this topic dropped overall (external demand change, not a content issue).

6. content_cb112fce36be — 309,910 impressions, position 5.6, CTR 0.16, declining. Would be wrong if this page recently changed intent/target keyword and the "decline" reflects a deliberate content pivot, not a problem.

7. content_9532f197bbc8 — 309,192 impressions, position 2.0, CTR 0.87, declining. Best position in this list with strong CTR — an unusual combination for "declining." Would be wrong if this is a false positive from the label itself (a proxy issue, not a real decline) since a page ranking this well with this CTR looks otherwise healthy.

8. content_008fb02c46cb — 236,803 impressions, position 4.4, CTR 0.26, declining. Would be wrong if this is noise — a single bad week pulling down the trend window rather than a persistent pattern.

9. content_813e88069237 — 233,561 impressions, position 26.2, CTR 0.06, declining. Much worse position tier (deep) — CTR here (0.06) is actually close to the deep-tier norm (~0.15), so this may be a position problem, not a CTR problem specifically. Would be wrong if position 26 is where this topic realistically belongs given competition, making "refresh" not the right action (may need a different strategy).

10. content_ff94c9b6b411 — 228,566 impressions, position 27.4, CTR 0.04, declining. Similar deep-tier case. Would be wrong for the same reason as #9 — low CTR here may simply track expected deep-tier behavior rather than signal an urgent problem.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest picks are #7 (content_9532f197bbc8) and #9/#10 (the two deep-tier pages). #7 is weak because it ranks well (position 2.0) with strong CTR (0.87) — properties of a healthy page, not a declining one — so its "declining" label may be a proxy artifact rather than a real problem, and flagging it for refresh could waste reviewer time on a page that's actually fine. #9 and #10 are weak because their low CTR is largely explained by their poor position tier (deep, expected CTR ~0.15) rather than an independent content problem — meaning "review for refresh" may be the wrong action; these pages might need a different strategy (e.g. targeting a more realistic keyword) rather than a content refresh.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.